**Deep Learning Framework for Leaf Disease Detection and Crop Damage Assessment**

In [23]:
import tensorflow as tf

print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))

Num GPUs Available:  0


In [22]:
!pip install tensorflow opencv-python scikit-learn matplotlib efficientnet transformers segmentation_models_pytorch

In [24]:
import numpy as np
print(np.__version__)


2.0.2


In [25]:
import tensorflow as tf
import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.applications import ResNet50, VGG16, EfficientNetB0, InceptionV3, DenseNet121
from keras.layers import Dense, Dropout, Flatten, GlobalAveragePooling2D
from keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import cv2
from keras.applications import imagenet_utils


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "c:\Users\HP\anaconda3\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "c:\Users\HP\anaconda3\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "c:\Users\HP\anaconda3\lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\HP\anaconda3\lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\HP\anaconda3\lib\site-packages\ipykernel

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [26]:
# Load dataset
data_dir = 'C:\\Users\\HP\\OneDrive\\Desktop\\project\\Chapter4 Data\\Code\\PlantVillage'  # Path to the PlantVillage dataset
image_size = (224, 224)

In [27]:
un_healthy_images_list = ['Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Target_Spot',
                                     'Tomato_Late_blight', 'Tomato_Bacterial_spot','Tomato__Tomato_mosaic_virus', 'Tomato_Septoria_leaf_spot',
                                     'Tomato_Early_blight', 'Tomato_Leaf_Mold']

un_healthy_images_list

['Tomato__Tomato_YellowLeaf__Curl_Virus',
 'Tomato__Target_Spot',
 'Tomato_Late_blight',
 'Tomato_Bacterial_spot',
 'Tomato__Tomato_mosaic_virus',
 'Tomato_Septoria_leaf_spot',
 'Tomato_Early_blight',
 'Tomato_Leaf_Mold']

In [28]:

from PIL import Image

def is_image_file(filename):
    filename = filename.lower()
    return filename.endswith(".jpg") or filename.endswith(".jpeg") or filename.endswith(".png")

def load_dataset(data_dir, image_size, main_dir="C:\\Users\\HP\\OneDrive\\Desktop\\project\\Chapter4 Data\\Code\\PlantVillage"):
    image_data = []
    # labels = []

    for dir in data_dir:
      path = os.path.join(main_dir, dir)
      # print(path)

      imageNames = os.listdir(path)
      # print(imageNames)
      for image in imageNames:
        if not is_image_file(image):
              continue  # Skip non-image files

        image_path = os.path.join(path, image)
        imgs = Image.open(image_path)
        img = np.array(imgs)

        if img is None:
          print(f"Warning: Failed to load image {image}")
          continue  # Skip invalid images

        img = cv2.resize(img, image_size)
        image_data.append(img)
        # labels.append(label)
      # Append the corresponding label

    return np.array(image_data)



In [30]:
    pip install numpy


Note: you may need to restart the kernel to use updated packages.


In [31]:
import cv2
from keras.applications import imagenet_utils

OpenCV bindings requires "numpy" package.
Install it via command:
    pip install numpy


ImportError: cannot import name 'fastCopyAndTranspose' from 'numpy.core._multiarray_umath' (c:\Users\HP\anaconda3\lib\site-packages\numpy\core\_multiarray_umath.py)

In [18]:
data_dir = un_healthy_images_list
image_size = (224, 224)  # Example size
unhealthy_images = load_dataset(data_dir, image_size)
# print(unhealthy_images)  # Prints the labels corresponding to each image

NameError: name 'cv2' is not defined

In [13]:
unhealthy_images.shape

(12743, 224, 224, 3)

In [14]:
np.random.seed(42)  # For reproducibility
indices = np.random.choice(len(unhealthy_images), 1000, replace=False)
unhealthy_images_resized = unhealthy_images[indices]
unhealthy_images_resized.shape

(1000, 224, 224, 3)

In [15]:
unhealthy_labels = np.ones(1000)


In [16]:
healthy_images_list = ["Tomato_healthy"]
healthy_images_list

['Tomato_healthy']

In [17]:
data_dir = healthy_images_list
# print(data_dir)
image_size = (224, 224)  # Example size
healthy_images = load_dataset(data_dir, image_size)
healthy_images

array([[[[132, 131, 137],
         [113, 112, 118],
         [ 99,  98, 104],
         ...,
         [116, 113, 124],
         [134, 131, 142],
         [120, 117, 128]],

        [[116, 115, 121],
         [119, 118, 123],
         [127, 126, 132],
         ...,
         [115, 112, 123],
         [112, 108, 120],
         [124, 121, 132]],

        [[139, 138, 144],
         [137, 136, 142],
         [130, 129, 135],
         ...,
         [111, 107, 118],
         [113, 110, 121],
         [124, 121, 132]],

        ...,

        [[ 69,  73,  77],
         [ 80,  82,  87],
         [ 67,  70,  75],
         ...,
         [ 83,  83,  93],
         [ 74,  74,  84],
         [ 78,  78,  88]],

        [[ 68,  71,  76],
         [ 66,  69,  74],
         [ 58,  61,  66],
         ...,
         [ 76,  76,  86],
         [ 78,  78,  88],
         [ 90,  90, 100]],

        [[ 70,  73,  78],
         [ 56,  59,  64],
         [ 56,  59,  64],
         ...,
         [ 75,  75,  85],
        

In [18]:
healthy_images.shape

(1591, 224, 224, 3)

In [19]:
healthy_labels = np.zeros(1591)
healthy_labels

array([0., 0., 0., ..., 0., 0., 0.])

In [20]:
images = np.concatenate((healthy_images, unhealthy_images_resized), axis=0)
labels = np.concatenate((healthy_labels, unhealthy_labels), axis=0)
print(images.shape)
print(labels.shape)

(2591, 224, 224, 3)
(2591,)


In [21]:
from sklearn.model_selection import train_test_split

In [22]:
images = images / 255.0


X_train, X_val, y_train, y_val = train_test_split(images, labels, test_size=0.2, random_state=42)

# RESNET

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split


In [23]:
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

# Load ResNet50 model
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
x = base_model.output
x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(2, activation='softmax')(x)

In [24]:
# Create the final model
model = Model(inputs=base_model.input, outputs=predictions)

# Freeze the layers of ResNet50
for layer in base_model.layers:
    layer.trainable = False

In [25]:
# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [27]:
# Train the model
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=10
)

Epoch 1/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 112s 2s/step - accuracy: 0.4955 - loss: 4.6031
Epoch 2/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 111s 2s/step - accuracy: 0.6119 - loss: 0.6890
Epoch 3/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 111s 2s/step - accuracy: 0.6163 - loss: 0.6819
Epoch 4/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 108s 2s/step - accuracy: 0.5997 - loss: 0.6795
Epoch 5/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 112s 2s/step - accuracy: 0.6278 - loss: 0.6702
Epoch 6/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 110s 2s/step - accuracy: 0.6221 - loss: 0.6700
Epoch 7/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 109s 2s/step - accuracy: 0.5945 - loss: 0.6755
Epoch 8/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 112s 2s/step - accuracy: 0.6086 - loss: 0.6702
Epoch 9/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 107s 2s/step - accuracy: 0.6177 - loss: 0.6663
Epoch 10/10
65/65 ━━━━━━━━━━━━━━━━━━━━ 111s 2s/step - accuracy: 0.6121 - loss: 0.6681


In [28]:
# Evaluate the model on the validation set
val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=1)
print(f"Test Accuracy: {val_accuracy}")

17/17 ━━━━━━━━━━━━━━━━━━━━ 26s 1s/step - accuracy: 0.6212 - loss: 0.6642
Test Accuracy: 0.6088631749153137


In [30]:
from sklearn.metrics import classification_report

# Generate classification report for test set
y_pred = model.predict(X_val)
y_pred_classes = np.argmax(y_pred, axis=1)
print(classification_report(y_val, y_pred_classes))

17/17 ━━━━━━━━━━━━━━━━━━━━ 23s 1s/step
              precision    recall  f1-score   support

         0.0       0.61      1.00      0.76       316
         1.0       0.00      0.00      0.00       203

    accuracy                           0.61       519
   macro avg       0.30      0.50      0.38       519
weighted avg       0.37      0.61      0.46       519



c:\Users\AJITHKUMAR\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\AJITHKUMAR\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\Users\AJITHKUMAR\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [44]:
pip install vit-keras


     ---------------------------------------- 43.5/43.5 kB ? eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [32]:
from transformers import ViTForImageClassification, ViTFeatureExtractor
import torch
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder

In [42]:
# Pretrained Vision Transformer
feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224')
vit_model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Concatenate
from vit_keras import vit

# Initialize ResNet50 model
resnet = ResNet50(include_top=False, weights='imagenet', input_shape=(224, 224, 3))
resnet.trainable = False  # Freeze the ResNet50 layers

# Initialize Vision Transformer model
vit_model = vit.vit_b32(image_size=224, pretrained=True, include_top=False)
vit_model.trainable = False  # Freeze the Vision Transformer layers

# Feature extraction
input_layer = tf.keras.layers.Input(shape=(224, 224, 3))
resnet_features = resnet(input_layer)
vit_features = vit_model(input_layer)

# Global average pooling
resnet_features = GlobalAveragePooling2D()(resnet_features)
vit_features = GlobalAveragePooling2D()(vit_features)

# Feature concatenation
concatenated_features = Concatenate()([resnet_features, vit_features])

# Classification head
dense_layer = Dense(1024, activation='relu')(concatenated_features)
output_layer = Dense(num_classes, activation='softmax')(dense_layer)

# Hybrid model
hybrid_model = tf.keras.Model(inputs=input_layer, outputs=output_layer)

# Compile the model
hybrid_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Summary
hybrid_model.summary()


In [35]:
from sklearn.preprocessing import LabelEncoder

# Initialize the LabelEncoder
label_encoder = LabelEncoder()

# Convert string labels in y_train and y_test to integers
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_val)


In [36]:
class CustomDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels  # Now these are numeric labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        # Apply feature extractor to the image
        encoding = feature_extractor(image, return_tensors="pt")

        # Extract pixel values for input
        pixel_values = encoding['pixel_values'].squeeze()  # Remove the batch dimension

        return pixel_values, torch.tensor(label, dtype=torch.long)  # Convert label to a long tensor


In [37]:
from transformers import ViTForImageClassification, ViTFeatureExtractor
import torch
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder

# Pretrained Vision Transformer
feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224')
vit_model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')

# Convert string labels to numeric using LabelEncoder
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)  # Encoded labels for training set
y_test_encoded = label_encoder.transform(y_val)  # Encoded labels for test set

# Define your Custom Dataset class
class CustomDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels  # These are now numeric labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        # Apply feature extractor to the image
        encoding = feature_extractor(image, return_tensors="pt")

        # Extract pixel values for input
        pixel_values = encoding['pixel_values'].squeeze()  # Remove the batch dimension

        return pixel_values, torch.tensor(label, dtype=torch.long)  # Return pixel values and numeric label



In [38]:
# Create Dataset and DataLoader
train_dataset = CustomDataset(X_train, y_train_encoded)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Move the model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vit_model.to(device)

# Define optimizer
optimizer = optim.Adam(vit_model.parameters(), lr=1e-4)

# Number of epochs.
epochs = 10

# Training Loop for Vision Transformer (ViT)
vit_model.train()  # Set model to training mode
for epoch in range(epochs):
    total_loss = 0.0
    for batch in train_loader:
        inputs, labels = batch  # Unpack batch

        # Move data to the same device as the model
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = vit_model(inputs, labels=labels)

        # Compute loss
        loss = outputs.loss
        total_loss += loss.item()

        # Backward pass
        loss.backward()

        # Update weights
        optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader)}")


KeyboardInterrupt: 

In [ ]:
# Move the model to GPU if available, else use CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vit_model = vit_model.to(device)  # Move model to device (GPU/CPU)

# Training Loop for Vision Transformer (ViT)
vit_model.train()  # Set model to training mode

# Define number of epochs
epochs = 10


for epoch in range(epochs):
    total_loss = 0.0
    for batch in train_loader:
        inputs, labels = batch  # Unpack batch

        # Move data (inputs and labels) to the same device as the model
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = vit_model(inputs, labels=labels)

        # Compute loss
        loss = outputs.loss
        total_loss += loss.item()

        # Backward pass
        loss.backward()

        # Update weights
        optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader)}")


Epoch 1/10, Loss: 0.15296490490436554
Epoch 2/10, Loss: 0.14053667290136218
Epoch 3/10, Loss: 0.14182622730731964
Epoch 4/10, Loss: 0.1447768872603774
Epoch 5/10, Loss: 0.14999751560389996
Epoch 6/10, Loss: 0.14487932156771421
Epoch 7/10, Loss: 0.14024454448372126
Epoch 8/10, Loss: 0.13736401265487075
Epoch 9/10, Loss: 0.13523170305415988
Epoch 10/10, Loss: 0.13798292260617018


In [ ]:
print(label_encoder.classes_)  # This will show the class names in the order they are encoded
print(y_train_encoded[:10])    # This will print the first 10 numeric labels


['Pepper__bell___healthy' 'Tomato_Late_blight' 'Tomato_healthy']
[1 1 1 1 1 1 1 1 1 1]


In [ ]:
# Move the model to GPU if available, else use CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
vit_model = vit_model.to(device)  # Move model to device (GPU/CPU)

# Training Loop for Vision Transformer (ViT)
vit_model.train()  # Set model to training mode

# Define number of epochs
epochs = 10

# Function to calculate accuracy
def calculate_accuracy(preds, labels):
    _, predicted = torch.max(preds, 1)  # Get the index of the max logit (predicted label)
    correct = (predicted == labels).sum().item()  # Count how many predictions are correct
    accuracy = correct / labels.size(0)  # Compute accuracy as the percentage of correct predictions
    return accuracy

for epoch in range(epochs):
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for batch in train_loader:
        inputs, labels = batch  # Unpack batch

        # Move data (inputs and labels) to the same device as the model
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = vit_model(inputs, labels=labels)

        # Compute loss
        loss = outputs.loss
        total_loss += loss.item()

        # Backward pass
        loss.backward()

        # Update weights
        optimizer.step()

        # Calculate accuracy for this batch
        batch_accuracy = calculate_accuracy(outputs.logits, labels)
        total_correct += batch_accuracy * labels.size(0)  # Accumulate correct predictions
        total_samples += labels.size(0)  # Accumulate the total number of samples

    # Epoch-wise average loss and accuracy
    epoch_loss = total_loss / len(train_loader)
    epoch_accuracy = total_correct / total_samples

    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}")


Epoch 1/10, Loss: 0.1417, Accuracy: 0.9758
Epoch 2/10, Loss: 0.1385, Accuracy: 0.9758
Epoch 3/10, Loss: 0.1331, Accuracy: 0.9758
Epoch 4/10, Loss: 0.1334, Accuracy: 0.9758
Epoch 5/10, Loss: 0.1296, Accuracy: 0.9758
Epoch 6/10, Loss: 0.1438, Accuracy: 0.9758
Epoch 7/10, Loss: 0.1324, Accuracy: 0.9758
Epoch 8/10, Loss: 0.1409, Accuracy: 0.9758
Epoch 9/10, Loss: 0.1379, Accuracy: 0.9758
Epoch 10/10, Loss: 0.1379, Accuracy: 0.9758


In [ ]:
!pip install tensorflow opencv-python-headless


In [ ]:
import tensorflow as tf

# Check if GPU is available
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [ ]:
import os

# Check if the models directory already exists
if not os.path.exists('models'):
    !git clone https://github.com/tensorflow/models.git
else:
    print("The 'models' directory already exists.")


The 'models' directory already exists.


In [ ]:
# Change directory to models/research
%cd models/research/

# Install protobuf, a dependency for Object Detection API
!apt-get install -y protobuf-compiler

# Compile the protobuf files
!protoc object_detection/protos/*.proto --python_out=.

# Install TensorFlow Object Detection API and other dependencies
!pip install -r requirements.txt
!pip install .

# Install COCO API, used for evaluation
!pip install pycocotools


/content/models/research
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
protobuf-compiler is already the newest version (3.12.4-1ubuntu7.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 49 not upgraded.
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
ERROR: Directory '.' is not installable. Neither 'setup.py' nor 'pyproject.toml' found.


In [ ]:
# Run this test to ensure that the installation is successful
!python object_detection/builders/model_builder_tf2_test.py


2024-10-22 09:57:19.903475: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-22 09:57:19.965720: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-22 09:57:19.992923: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-22 09:57:23.896866: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
Traceback (most recent call last):
  File "/content/models/research/object_detection/builders/model_builder_tf2_test.py", line 24, in <module>
    from object_detection.builders import model_builder
ModuleNotFoundError: No module named 'obje

In [ ]:
from object_detection.utils import config_util
from tensorflow.keras.layers import Rescaling, RandomFlip, RandomRotation


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Rescaling, RandomFlip, RandomRotation, Conv2D, MaxPooling2D, Flatten, Dense

# Define a simple CNN model with preprocessing layers
model = Sequential([
    Rescaling(1./255, input_shape=(224, 224, 3)),
    RandomFlip('horizontal'),
    RandomRotation(0.1),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(),
    Flatten(),
    Dense(128, activation='relu'),
    Dense(10, activation='softmax')  # Assuming 10 classes for classification
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


/usr/local/lib/python3.10/dist-packages/keras/src/layers/preprocessing/tf_data_layer.py:19: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
